# Accessing Data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Holiday Function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

#Monarch's Birthday
def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]   # Monday = 0
    return mondays[1]                   # second Monday



# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

HOLIDAY_GROUPS = {
    "Good Friday": "Easter Long Weekend",
    "Easter Saturday": "Easter Long Weekend",
    "Easter Sunday": "Easter Long Weekend",
    "Easter Monday": "Easter Long Weekend",

    "Christmas Day": "Christmas and Boxing Day",
    "Boxing Day": "Christmas and Boxing Day",
}



In [ ]:
print(HOLIDAYS_VIC)


# Lat/lon information

In [ ]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="sydney_demand_mapper")

def get_coords(place):
    """Return (lat, lon) for a suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None

# Apply geocoding to the 'Name' column
latitudes, longitudes = [], []
for suburb in info['Name']:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # polite pause to avoid hitting API limits

info['latitude'] = latitudes
info['longitude'] = longitudes

In [ ]:
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()
#Dee Why West doesn't exist as a suburb polygon, so will need to change the name to Dee Why

In [ ]:
dee_why_west_lat = -33.73441
dee_why_west_lon = 151.28278
#Found the lat/lon information online

In [ ]:
info.loc[info["Name"] == "Dee Why West", "latitude"] = dee_why_west_lat
info.loc[info["Name"] == "Dee Why West", "longitude"] = dee_why_west_lon
#inputting lat and lon from online into info

In [ ]:
info.head()

# Creating new csv file
- One row per day
- 30 days before the holiday
- the holiday itself
- 30 days after the holiday
- So 61 rows per holiday × year × station
- also ensures new year's day and christmas day include any days within the 30 +/- days that are not in the same year
- includes columns defining what the day's name is (Monday, Tuesday etc), and if it's a weekend (True/False)
----------------------------------
- For each day, the function will compute:
- Block‑level mean, standard deviation, and variance of the relative‑rank values for that day’s 24‑hour profile.
---------------------------------
- Using new time blocks:
- 04–10
- 10–15
- 15–20
- 20–24
- 00–04
---------------------------------
- Using your 2‑year forward‑looking ranking window (Y + Y+1)
- metadata integrated (station id, station name, residential fraction, industrial fraction, dwellings, persons)

** UPDATE 10_03_26
- now includes lat and lon columns of the approximate location of each substation (based on suburb location)
- created a new column for 2 'new' public holidays, which combine pre-existing holidays as they are very similar. All easter related holidats => Easter Long Weekend, Christmas Day and Boxing Day => Christmas and Boxing Day

In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import os


def compute_two_year_daily_relative_rank_csv(
    demand,
    holiday_lib,
    info,
    holiday_groups=None,
    window_days=30,
    blocks=None,
    out_csv="full_nsw_relative_rank.csv"
):
    """
    Compute daily block-level mean, std, and variance of relative ranks
    for ±window_days around each holiday, for each year and station.

    Ranking pool = all hourly data from year Y and Y+1.
    Window extraction allows spillover across years.

    Output = one row per station × holiday × year × day.
    """

    # -----------------------------
    # Validate inputs
    # -----------------------------
    if blocks is None:
        raise ValueError("You must supply a dictionary of time blocks.")

    if holiday_groups is None:
        holiday_groups = {}

    # -----------------------------
    # Ensure datetime index
    # -----------------------------
    demand.index = pd.to_datetime(demand.index)

    # Hourly mean demand
    hourly = demand.resample("h").mean()

    # All station columns
    stations = [c for c in hourly.columns if c not in ["date", "hour"]]

    years = list(range(2004, 2018))

    # -----------------------------
    # Unified progress bar
    # -----------------------------
    total_iterations = len(holiday_lib) * len(years) * len(stations)
    pbar = tqdm(total=total_iterations, desc="Processing all holiday-year-station combos")

    rows = []

    # ============================================================
    # LOOP HOLIDAYS
    # ============================================================
    for holiday_name, holiday_func in holiday_lib.items():

        for year in years:

            ref_date = holiday_func(year)

            # --------------------------------------------------------
            # Expanded pool for spillover window extraction
            # --------------------------------------------------------
            pool_start = pd.Timestamp(f"{year}-01-01") - pd.Timedelta(days=31)
            pool_end   = pd.Timestamp(f"{year+1}-12-31") + pd.Timedelta(days=31)

            expanded_pool = hourly.loc[pool_start:pool_end]
            if expanded_pool.empty:
                # Still update progress for all stations
                pbar.update(len(stations))
                continue

            # --------------------------------------------------------
            # Extract ±window_days around holiday
            # --------------------------------------------------------
            win_start = ref_date - pd.Timedelta(days=window_days)
            win_end   = ref_date + pd.Timedelta(days=window_days)

            window = expanded_pool.loc[win_start:win_end].copy()
            if window.empty:
                pbar.update(len(stations))
                continue

            window["date"] = window.index.date
            window["hour"] = window.index.hour

            # --------------------------------------------------------
            # Ranking pool = Y + Y+1 only
            # --------------------------------------------------------
            rank_start = pd.Timestamp(f"{year}-01-01")
            rank_end   = pd.Timestamp(f"{year+1}-12-31")

            ranking_pool = hourly.loc[rank_start:rank_end]

            # ============================================================
            # LOOP STATIONS
            # ============================================================
            for station in stations:

                # Update unified progress bar
                pbar.update(1)

                if station not in window.columns:
                    continue

                W = window.copy()

                # --------------------------------------------------------
                # Compute relative rank per hour
                # --------------------------------------------------------
                rp = ranking_pool[[station]].copy()
                rp["date"] = rp.index.date
                rp["hour"] = rp.index.hour

                rp["rank"] = rp.groupby("hour")[station].rank(method="average")
                n_days = rp.groupby("hour")["date"].transform("nunique")
                rp["relative_rank"] = rp["rank"] / n_days

                # Merge relative ranks back into window
                W = W.merge(
                    rp[["relative_rank", "hour", "date"]],
                    on=["hour", "date"],
                    how="left"
                )

                # --------------------------------------------------------
                # Metadata from info (including lat/lon)
                # --------------------------------------------------------
                station_code = station
                station_name = info.loc[station_code, "Name"]

                residential = info.loc[station_code, "Residential"]
                industrial  = info.loc[station_code, "Industrial"]
                dwellings   = info.loc[station_code, "Dwellings"]
                persons     = info.loc[station_code, "Persons"]

                # Robust lat/lon lookup
                lat = info.loc[station_code].get("latitude") \
                      or info.loc[station_code].get("Latitude") \
                      or info.loc[station_code].get("lat")

                lon = info.loc[station_code].get("longitude") \
                      or info.loc[station_code].get("Longitude") \
                      or info.loc[station_code].get("lon")

                # Combined holiday label
                holiday_group = holiday_groups.get(holiday_name, holiday_name)

                # ============================================================
                # LOOP DAYS
                # ============================================================
                for day in sorted(W["date"].unique()):

                    mask = W["date"] == day
                    day_rr = W.loc[mask, "relative_rank"]

                    # Always keep the holiday day
                    is_holiday_day = (pd.Timestamp(day).date() == ref_date.date())

                    # Baseline days must have ≥18 hours
                    if (day_rr.shape[0] < 18) and (not is_holiday_day):
                        continue

                    # Reindex to hour 0–23
                    day_rr.index = W.loc[mask, "hour"]

                    weekday_name = pd.Timestamp(day).day_name()
                    is_weekend = weekday_name in ["Saturday", "Sunday"]

                    # --------------------------------------------------------
                    # Block statistics
                    # --------------------------------------------------------
                    block_stats = {}
                    for block_name, hours in blocks.items():
                        vals = day_rr.loc[list(hours)]
                        block_stats[f"{block_name}_mean"] = vals.mean()
                        block_stats[f"{block_name}_std"]  = vals.std()
                        block_stats[f"{block_name}_var"]  = vals.var()

                    # --------------------------------------------------------
                    # Append row
                    # --------------------------------------------------------
                    rows.append({
                        "year": year,
                        "holiday": holiday_name,
                        "holiday_group": holiday_group,
                        "is_holiday": is_holiday_day,

                        "station_code": station_code,
                        "station_name": station_name,
                        "lat": lat,
                        "lon": lon,

                        "date": pd.Timestamp(day),
                        "weekday_name": weekday_name,
                        "is_weekend": is_weekend,

                        **block_stats,

                        "residential": residential,
                        "industrial": industrial,
                        "persons": persons,
                        "dwellings": dwellings,
                    })

    # Close progress bar
    pbar.close()

    # ============================================================
    # Build final DataFrame
    # ============================================================
    df = pd.DataFrame(rows).round(4)

    # Column ordering
    block_cols = [c for c in df.columns if any(s in c for s in ["_mean", "_std", "_var"])]

    ordered_cols = (
        ["year", "holiday", "holiday_group", "is_holiday"] +
        ["station_code", "station_name", "lat", "lon",
         "date", "weekday_name", "is_weekend"] +
        block_cols +
        ["residential", "industrial", "persons", "dwellings"]
    )

    df = df[ordered_cols]

    # Save
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    df.to_csv(out_csv, index=False)

    return df


In [ ]:
blocks = {
    "04_10": range(4, 10),
    "10_15": range(10, 15),
    "15_20": range(15, 20),
    "20_24": range(20, 24),
    "00_04": range(0, 4),
}


In [ ]:
df = compute_two_year_daily_relative_rank_csv(
    demand=demand,
    holiday_lib=HOLIDAYS_VIC,
    info=info,
    blocks=blocks,
    out_csv="/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"
)
